### Cuaderno de simulaciones con Schelling 3 agentes

Cargamos los paquetes necesarios (en mi portatel se debe ejecutar en el entorno mesa-env en WSL)

In [ ]:
!pip install mesa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 272.3/272.3 kB 2.9 MB/s eta 0:00:00


In [ ]:
import mesa
import random
import time
import pandas as pd
from openpyxl import Workbook, load_workbook
import os
import gc
#from mesa.visualization import SolaraViz, make_space_component
#from ipywidgets import interact, IntSlider, FloatSlider
#import solara
#rom IPython.display import display, HTML

# Cargamos las funcines del modelo de Schelling con 3 poblaciones

En esta celda cargamos el agente **Persona** y sus funciones (**incomodidad**, **move**)

In [ ]:
# ========== AGENTE ==========
class Persona(mesa.Agent):
    def __init__(self, model, tipo: int, se_ha_movido: bool) -> None:
        super().__init__(model)
        self.tipo = tipo  # tipos (-1), (+1), hostiles y 0 neutrales
        self.se_ha_movido = se_ha_movido # para actualizar TRUE cuando el agente cambia de posición


# La incomodidad es de un agente (self) en una casilla (pos) asumiendo que el agente es de un tipo. incomodidad(agente,posición,tipo) es el número
# de vecinos incómodos que tendría el agente si se encontrara en la posición.

    def incomodidad(self, pos, tipo):
        contador = 0
        vecindad = self.model.grid.get_neighbors(pos, moore=True, include_center=False)
        if tipo == 1:
            for vecino in vecindad:
                if vecino.tipo == -1: contador += 1
        elif tipo == -1:
            for vecino in vecindad:
                if vecino.tipo == 1: contador += 1
        elif tipo == 0:
            for vecino in vecindad:
                contador += abs(vecino.tipo)
        return contador

# Esta es la función fundamental que recoloca a un agente, siempre que su incomodidad esté por encima de su tolerancia.
# Para ello busca la casilla vacía más cercana en la que tendría una incomodidad menor a la actual, y se mueve ahí.
# Si no encuentra ninguna casilla con mejor situación, se resigna y se queda donde está.

    def move(self):
        self.se_ha_movido = False
        possible_steps = []
        radio = 0
        max_iteraciones = self.model.grid.width // 2 + 1
        incomodidad_base = self.incomodidad(self.pos, self.tipo)

        if incomodidad_base > self.model.tolerance:
            while not possible_steps and radio < max_iteraciones:
                radio += 1
                vecindario = self.model.grid.get_neighborhood(
                    self.pos, moore=True, include_center=False, radius=radio
                )
                empty_steps = [step for step in vecindario if self.model.grid.is_cell_empty(step)]
                possible_steps = [step for step in empty_steps if self.incomodidad(step, self.tipo) < incomodidad_base]

        if possible_steps:
            new_position = random.choice(possible_steps)
            self.model.grid.move_agent(self, new_position)
            self.se_ha_movido = True




En esta celda definimos el modelo

  Modelo_Desplazamiento(N1,N2,N3, width, height, tol)

  N1 := población tipo (+1)
  N2 := población tipo (0)
  N3 := población tipo (-1)
  tol := los agentes se desplazan cuando incomodidad > tol

  **contact_measure**

  
  

In [ ]:
# ========== MODELO ==========

class Modelo_Desplazamiento(mesa.Model):
    def __init__(self, N1, N2, N3, width, height, tol, seed=None):
        super().__init__(seed=seed)
        self.num_agents = N1 + N2 + N3
        self.grid = mesa.space.SingleGrid(width, height, True)
        self.running = True
        self.tolerance = tol
        self.steps_to_equilibrium = 0
        self.in_equilibrium = False
        self.contact_measure = {}
        self.datacollector = mesa.DataCollector(
            model_reporters={
                "Steps_to_Equilibrium": "steps_to_equilibrium",
                "Contact_Measure": "contact_measure",
                "In_Equilibrium": "in_equilibrium"
            }
        )

        for i in range(N1): Persona(model=self, tipo=1, se_ha_movido=False)
        for i in range(N2): Persona(model=self, tipo=0, se_ha_movido=False)
        for i in range(N3): Persona(model=self, tipo=-1, se_ha_movido=False)


        # Obtener todas las posiciones del grid
        all_positions = [
            (x, y)
            for x in range(self.grid.width)
            for y in range(self.grid.height)
        ]

        # Mezclar aleatoriamente
        self.random.shuffle(all_positions)

        # Asignar una posición por agente
        for agent, pos in zip(self.agents, all_positions):
            self.grid.place_agent(agent, pos)

        self.calculate_contact_measure()

    def calculate_contact_measure(self):
        """Calcula la medida de contacto desagregada por tipo de agente"""
        contact_counts = {
            -1: {-1: 0, 0: 0, 1: 0},
             0: {-1: 0, 0: 0, 1: 0},
             1: {-1: 0, 0: 0, 1: 0},
        }
        Total_contacts = 0

        for agent in self.agents:
            neighbors = self.grid.get_neighbors(agent.pos, moore=True, include_center=False)
            for neighbor in neighbors:
                contact_counts[agent.tipo][neighbor.tipo] += 1
                Total_contacts += 1

        Total_contacts = Total_contacts / 2

        for i in [-1,0,1]: contact_counts[i][i] = contact_counts[i][i] / 2

        # Guardamos proporciones
        self.contact_measure = {}
        for agent_type in [-1, 0, 1]:
            self.contact_measure[agent_type] = {}
            for neighbor_type in [-1, 0, 1]:
                if Total_contacts > 0:
                    self.contact_measure[agent_type][neighbor_type] = (
                        contact_counts[agent_type][neighbor_type] / Total_contacts
                    )
                else:
                    self.contact_measure[agent_type][neighbor_type] = 0

    def check_equilibrium(self) -> bool:
        """Verifica si el modelo ha alcanzado el equilibrio"""
        for agent in self.agents:
            if agent.se_ha_movido:
                return False
        return True

    def step(self):
        if not self.in_equilibrium:
            self.agents.shuffle_do("move")
            self.calculate_contact_measure()

            if self.check_equilibrium():
                self.in_equilibrium = True
                self.running = False
            else:
                self.steps_to_equilibrium += 1

            self.datacollector.collect(self)

In [ ]:
def agentes_insatisfechos(model):
    """Cuenta los agentes insatisfechos por tipo en el modelo."""
    insatisfechos = { -1:0, 0:0, 1:0 }
    for agent in model.agents:
        if agent.incomodidad(agent.pos, agent.tipo) > model.tolerance:
            insatisfechos[agent.tipo] += 1
    return insatisfechos

Simulación

In [ ]:
# ------------------------------------------------------------
# Montecarlo simple
# ------------------------------------------------------------

def run_montecarlo(n_runs=25, N1=450, N2=600, N3=450, tol=2.0, width=40, height=40):
    resultados = []

    for run in range(1, n_runs+1):
        tiempo_inicio = time.time()
        model = Modelo_Desplazamiento(N1, N2, N3, width, height, tol, seed=run)

        # Ejecutar hasta equilibrio
        while model.steps_to_equilibrium < 20 and model.running and not model.in_equilibrium:
            model.step()

        # Medida de contacto final
        cm = model.contact_measure
        insatisfechos = agentes_insatisfechos(model)

        resultados.append({
            "Run": run,
            "Steps_to_Equilibrium": model.steps_to_equilibrium,
            # Contactos (6 medidas independientes)
            "C(-1,-1)": cm[-1][-1],
            "C(-1,0)": cm[-1][0],
            "C(-1,1)": cm[-1][1],
            "C(0,0)": cm[0][0],
            "C(0,1)": cm[0][1],
            "C(1,1)": cm[1][1],
            # Insatisfechos
            "Unsat_-1": insatisfechos[-1],
            "Unsat_0": insatisfechos[0],
            "Unsat_1": insatisfechos[1],
        })


        tiempo_fin = time.time()
        duracion = tiempo_fin - tiempo_inicio

        print(f"Experimento {run}/{n_runs} completado en {duracion:.2f} segundos.\n")

        model.running = False
        model.grid = None
        model.schedule = None
        model.datacollector = None
        del model
        gc.collect()
    # Pasar a DataFrame y exportar

    return pd.DataFrame(resultados)
    #df.to_excel(salida, index=False)



In [ ]:
# ------------------------------------------------------------
# Barrido sobre N2
# ------------------------------------------------------------


def barrido_N2(
    n_runs=20, N2_min=600, N2_max=1000, paso=2, total=1500,
    tol=2.0, width=40, height=40, salida="barrido.xlsx"
):
    # Crear archivo si no existe
    if not os.path.exists(salida):
        wb = Workbook()
        ws = wb.active
        ws.title = "Resultados"

        # Encabezados básicos
        columnas = ["N1", "N2", "N3","STE_mean","STE_std","CAA_mean","CAA_std","CAB_mean","CAB_std","CAC_mean","CAC_std", "CBB_mean","CBB_std","CBC_mean","CBC_std","CCC_mean","CCC_std","UnsatA_mean","UnsatA_std","UnsatB_mean","UnsatB_std","UnsatC_mean", "UnsatC_std"]

        ws.append(columnas)
        wb.save(salida)



    print(f"\n=== Ejecutando barrido desde N2={N2_min}, N2={N2_max} ===")
    # Abrir libro para agregar filas
    for N2 in range(N2_min, N2_max + 1, paso):
        N1 = (total - N2) // 2
        N3 = N1

        print(f"\n=== Montecarlo para N2={N2} ===")

        df = run_montecarlo(n_runs, N1, N2, N3, tol, width, height)

        # crear fila
        fila = [N1, N2, N3]

        for col in df.columns:
            if col != "Run":
                fila.append(df[col].mean())
                fila.append(df[col].std())

        # Escribir fila en Excel
        wb = load_workbook(salida)
        ws = wb.active
        ws.append(fila)
        wb.save(salida)

        # Limpiar memoria
        del df
        del fila
        wb.close()

        gc.collect()

    print(f"\n✔ Barrido completado. Guardado en {salida}")


Este código hace un barrido por valores de los parámetros N2 en un rango quedandose con las estadísticas

In [ ]:
 barrido_N2(
        n_runs=25,
        N2_min=100,
        N2_max=600,
        paso=2,
        salida="barridoN2.xlsx"
    )